In [1]:
import pandas as pd

from langchain.chat_models import ChatOpenAI
from langchain.output_parsers import PydanticOutputParser
from langchain.pydantic_v1 import BaseModel, Field
from langchain.schema import (HumanMessage, SystemMessage)

from dotenv import load_dotenv
load_dotenv()

chat = ChatOpenAI(temperature=0, model_name='gpt-4', request_timeout=120) # gpt-3.5-turbo-16k

/Users/shreyas.sk/PycharmProjects/Skeptic-SEBI/venv/lib/python3.9/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
df = pd.read_csv("data/combined_video_info_utf_16.csv", encoding='utf-16')

In [3]:
df.head(5)

,video_id,title,author,description,org_transcript,eng_transcript,image,url
0,flIG8Lw34Cw,D 30 Strategy | Make Daily 5000 to 10k risk Free,Baap of Chart,Today we will learn about the D30 Option Buyi...,वो हार्फ़ेंट कैसे आप लोग अच्छे होंगे आपलोग कमे...,"Text: ""How will you guys be good at Harfent? Y...",<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=flIG8Lw34Cw
1,ynfEvP5kYCk,Rs 3000 Daily Income from Stock Market | Using...,THE CATALYST GROUP,Earn money daily from stock market | Trick to ...,"Hello everybody, kyaal chaal hai, this is A.S....","Hello everybody, how are you doing? This is A....",<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=ynfEvP5kYCk
2,28Jay-3S8fg,Supply Demand Strategy | Trade Swing | Intrada...,Trade Swings,✅ Download NOW ✅ Trade Swings Application (Fre...,दोस्तों आप दिica सब्तू याहापं आप ओंड को बाई कर...,"Friends, you all know that here on Diksha, you...",<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=28Jay-3S8fg
3,BnZU6qYVUl0,99% Accurate | Trade Swing | Intraday Trading ...,Trade Swings,✅ Download NOW ✅ Trade Swings Application (Fre...,"बेन्ग नेफ्टी की यही पढ़टेजिया है, आप देख समकते...",This text seems to be a mixture of Hindi and a...,<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=BnZU6qYVUl0
4,swst4yk-ow8,Bank Bees 100% Risk Free Investment | 1 लाख से...,Dr. Mukul Agrawal,For daily stock market updates join our Telegr...,Hello everyone how are you all and what's goi...,"""Hello everyone, how are you all and what's go...",<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=swst4yk-ow8


In [8]:
class JSONFormatter(BaseModel):
   SummarizedClaims : str = Field(description="summary of claims made by the finfluencer")
   Justification : str = Field(description="justification provided by gpt-4 for the claims")

parser = PydanticOutputParser(pydantic_object=JSONFormatter)

def extract_claims_and_justification(sample):
    transcript = sample["eng_transcript"]
    messages = [
        SystemMessage(content="You are a Financial Analyst. You are provided with the youtube transcript of a fincancial influencer. \
                    Your task is to identify and extract the Claims/Tips/Tricks made by the Financial Influencer. \
                    Summarize the extracted claims into a small paragraph not exceeding 150 words. \
                    Extract all false claims and provide counter thesis or Justification for false claims. \
                    Please separate out the Summarized Claims and Justification from the response. \
                    Provide the response in JSON format containing keys for SummarizedClaims and Justification"),
        HumanMessage(content=transcript)
    ]
    response = chat(messages)
    formatted_output = parser.parse(response.content)
    return [formatted_output.SummarizedClaims, formatted_output.Justification]

In [9]:
df[["Summary_Claims", "Justification"]] = df.apply(extract_claims_and_justification, axis=1, result_type="expand")

Retrying langchain.chat_models.openai.ChatOpenAI.completion_with_retry.<locals>._completion_with_retry in 4.0 seconds as it raised APIConnectionError: Error communicating with OpenAI: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')).


In [11]:
# df.to_csv("data/claims_justification_data.csv", index=False)

In [10]:
df.head()

,video_id,title,author,description,org_transcript,eng_transcript,image,url,Summary_Claims,Justification
0,flIG8Lw34Cw,D 30 Strategy | Make Daily 5000 to 10k risk Free,Baap of Chart,Today we will learn about the D30 Option Buyi...,वो हार्फ़ेंट कैसे आप लोग अच्छे होंगे आपलोग कमे...,"Text: ""How will you guys be good at Harfent? Y...",<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=flIG8Lw34Cw,The financial influencer claims that with a ca...,The influencer's claim that one can easily mak...
1,ynfEvP5kYCk,Rs 3000 Daily Income from Stock Market | Using...,THE CATALYST GROUP,Earn money daily from stock market | Trick to ...,"Hello everybody, kyaal chaal hai, this is A.S....","Hello everybody, how are you doing? This is A....",<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=ynfEvP5kYCk,"The financial influencer, A.S. Pandit, claims ...",The claim that one can consistently earn a dai...
2,28Jay-3S8fg,Supply Demand Strategy | Trade Swing | Intrada...,Trade Swings,✅ Download NOW ✅ Trade Swings Application (Fre...,दोस्तों आप दिica सब्तू याहापं आप ओंड को बाई कर...,"Friends, you all know that here on Diksha, you...",<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=28Jay-3S8fg,The financial influencer suggests that profits...,The influencer's claim that profits can be mad...
3,BnZU6qYVUl0,99% Accurate | Trade Swing | Intraday Trading ...,Trade Swings,✅ Download NOW ✅ Trade Swings Application (Fre...,"बेन्ग नेफ्टी की यही पढ़टेजिया है, आप देख समकते...",This text seems to be a mixture of Hindi and a...,<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=BnZU6qYVUl0,The financial influencer suggests a trading st...,The influencer's claims are based on his perso...
4,swst4yk-ow8,Bank Bees 100% Risk Free Investment | 1 लाख से...,Dr. Mukul Agrawal,For daily stock market updates join our Telegr...,Hello everyone how are you all and what's goi...,"""Hello everyone, how are you all and what's go...",<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=swst4yk-ow8,"The financial influencer, Mukul Agrawal, promo...",The claim that Bank Bees offers a guaranteed 1...
